# Few-shot Prompt Baseline



# 1. Config

In [1]:
from uretriever.configs.api_config import get_env
from uretriever.configs.path_config import DATA_DIR, MODEL_DIR, PROMPT_DIR


MODEL_CONFIG = {
    "provider": "llama_cpp",
    "model_path": str(MODEL_DIR / "Mistral-7B-Instruct-v0.3.Q4_K_M.gguf"),
    "max_tokens": 512,
    "n_ctx": 4096,
    "temperature": 0,
    "n_threads": 4,
    "n_gpu_layers": -1,
    'verbose': False

    # "provider": "openai",
    # "model": "gpt-4.1",
    # "base_url": get_env("GITHUB_MODEL_API"),
    # "api_key": get_env("GITHUB_TOKEN"),
    # "max_tokens": 512,
    # "temperature": 0,
    # 'verbose': False
}

RAW_DATA_DIR = DATA_DIR / "raw"
print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"PROMPT_DIR: {PROMPT_DIR}")

RAW_DATA_DIR: D:\LegalRetrievalAgent\data\raw
PROMPT_DIR: D:\LegalRetrievalAgent\prompts


# 2. Setup

In [2]:
# Load chat model
from uretriever.chat_model_loader import load_chat_model


provider = MODEL_CONFIG["provider"]
model_config = {k: v for k, v in MODEL_CONFIG.items() if k != "provider"}
chat_model = load_chat_model(provider, **model_config)

llama_new_context_with_model: n_batch is less than GGML_KQ_MASK_PAD - increasing to 32
llama_new_context_with_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [3]:
# Load prompt
import json

from uretriever.utils import load_text, load_json


system_prompt = load_text(PROMPT_DIR / "system.txt")
examples = load_json(PROMPT_DIR / "examples.json")

print("\n-- system_prompt --\n")
print(system_prompt)

print("\n-- examples --\n")
print(json.dumps(examples, ensure_ascii=False, indent=2))


-- system_prompt --

You are a Swiss legal retrieval assistant. 
Your task is to find the most relevant Swiss legal citations for the user's question.

Output format:
- Return only a valid JSON array of strings.

Rules:
- Use standard Swiss citation style: 
  Federal laws: "Art. X ABBREV" where ABBREV is ZGB, OR, StGB, BV, etc.
  Court decisions: "BGE X Y Z" or "BGE X Y Z E. N" with consideration number
- Do not invent citations.


-- examples --

[
  {
    "query": "What are the requirements for a valid contract under Swiss law?",
    "answer": [
      "Art. 1 OR",
      "Art. 11 OR",
      "Art. 12 OR",
      "BGE 119 II 449 E. 2",
      "BGE 127 III 248 E. 3.1"
    ]
  },
  {
    "query": "When can a marriage be annulled in Switzerland?",
    "answer": [
      "Art. 104 ZGB",
      "Art. 105 ZGB",
      "Art. 106 ZGB",
      "BGE 121 III 38 E. 2b"
    ]
  },
  {
    "query": "What constitutes negligent homicide under Swiss criminal law?",
    "answer": [
      "Art. 117 StGB",
    

# 3. Generate citations

In [4]:
import json

from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate


def build_fewshot_prompt(system_prompt: str, examples: list[dict]) -> ChatPromptTemplate:
    example_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "{query}"),
            ("ai", "{answer}"),
        ]
    )

    few_shot_prompt = FewShotChatMessagePromptTemplate(
        examples=[
            {
                "query": ex["query"],
                "answer": json.dumps(ex["answer"], ensure_ascii=False),
            }
            for ex in examples
        ],
        example_prompt=example_prompt,
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            few_shot_prompt,
            ("human", "{query}"),
        ]
    )

    return prompt


prompt = build_fewshot_prompt(system_prompt, examples)


messages = prompt.format_messages(
    query="What are the requirements for a valid contract under Swiss law?"
)

for m in messages:
    print(type(m).__name__, "=>", m.content)
    print("---")



SystemMessage => You are a Swiss legal retrieval assistant. 
Your task is to find the most relevant Swiss legal citations for the user's question.

Output format:
- Return only a valid JSON array of strings.

Rules:
- Use standard Swiss citation style: 
  Federal laws: "Art. X ABBREV" where ABBREV is ZGB, OR, StGB, BV, etc.
  Court decisions: "BGE X Y Z" or "BGE X Y Z E. N" with consideration number
- Do not invent citations.

---
HumanMessage => What are the requirements for a valid contract under Swiss law?
---
AIMessage => ["Art. 1 OR", "Art. 11 OR", "Art. 12 OR", "BGE 119 II 449 E. 2", "BGE 127 III 248 E. 3.1"]
---
HumanMessage => When can a marriage be annulled in Switzerland?
---
AIMessage => ["Art. 104 ZGB", "Art. 105 ZGB", "Art. 106 ZGB", "BGE 121 III 38 E. 2b"]
---
HumanMessage => What constitutes negligent homicide under Swiss criminal law?
---
AIMessage => ["Art. 117 StGB", "Art. 12 StGB", "BGE 116 IV 306 E. 1a"]
---
HumanMessage => What are the grounds for divorce in Swiss 

In [5]:
from uretriever.citation import extract_citations


def generate_citations(chat_model, prompt, query, verbose=0):
    chain = prompt | chat_model
    response = chain.invoke({"query": query})
    output = response.content.strip()
    citations = extract_citations(output)
    
    if verbose == 1:
        print("\n-- human --\n")
        print(query)
        print("\n-- ai --\n")
        print(output)

    return citations

In [6]:
# Test on a example
import pandas as pd

from uretriever.citation import parse_citations


valid_df = pd.read_csv(RAW_DATA_DIR / "val.csv")
query = "What are the requirements for a valid contract under Swiss law?"
pred_citations = generate_citations(chat_model, prompt, query, verbose=1)
gold_citations = parse_citations(valid_df["gold_citations"].iloc[0])

true_positives = set(pred_citations) & set(gold_citations)
false_positives = set(pred_citations) - set(gold_citations)
false_negatives = set(gold_citations) - set(pred_citations)

print("\n-- pred_citations --\n")
print(len(pred_citations))
print(pred_citations)
print("\n-- gold citations --\n")
print(len(gold_citations))
print(gold_citations)
print("\n-- true positives --\n")
print(len(true_positives))
print(true_positives)
print("\n-- false positives --\n")
print(len(false_positives))
print(false_positives)
print("\n-- false negatives --\n")
print(len(false_negatives))
print(false_negatives)


-- human --

What are the requirements for a valid contract under Swiss law?

-- ai --

["Art. 1 OR", "Art. 11 OR", "Art. 12 OR", "BGE 119 II 449 E. 2", "BGE 127 III 248 E. 3.1"]

-- pred_citations --

5
['Art. 1 OR', 'Art. 11 OR', 'Art. 12 OR', 'BGE 119 II 449 E. 2', 'BGE 127 III 248 E. 3.1']

-- gold citations --

42
['Art. 221 Abs. 1 StPO', 'Art. 140 Abs. 1 StGB', 'Art. 396 Abs. 1 StPO', 'Art. 222 StPO', 'Art. 393 Abs. 1 StPO', 'Art. 382 Abs. 1 StPO', 'Art. 385 Abs. 1 StPO', 'Art. 221 Abs. 2 StPO', 'Art. 227 Abs. 1 StPO', 'Art. 212 Abs. 3 StPO', 'Art. 390 Abs. 2 StPO', 'Art. 422 Abs. 1 StPO', 'Art. 422 Abs. 2 StPO', 'Art. 428 Abs. 1 StPO', 'Art. 135 Abs. 4 StPO', 'Art. 100 Abs. 1 BGG', 'Art. 135 Abs. 3 StPO', 'Art. 37 Abs. 1 StBOG', 'Art. 39 Abs. 1 StBOG', 'BGE 137 IV 122 E. 6.2', 'BGE 137 IV 122 E. 6.4', 'BGE 137 IV 122 E. 4.2', 'BGE 132 I 21 E. 3.2', '1B_210/2023 E. 4.1', 'BGE 132 I 21 E. 3.2.2', '1B_536/2018 E. 5.1', 'BGE 139 IV 270 E. 3.1', 'BGE 133 I 168 E. 4.1', 'BGE 143 IV 1

# 4. Evaluation

In [7]:
valid_df = pd.read_csv(RAW_DATA_DIR / "val.csv")
valid_df.head()

,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...
3,val_004,"Mr. Dalton, born in 1941 and resident in a sma...",Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....
4,val_005,"A parent, separated from their co-parent since...",Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...


In [8]:
from tqdm.notebook import tqdm
import time


def generate_pred_df(df: pd.DataFrame, chat_model, prompt, provider: str) -> pd.DataFrame:
    preds = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating predictions"):
        query_id = row["query_id"]
        query = row["query"]

        citations = generate_citations(chat_model, prompt, query)
        citations_str = ";".join(citations) if citations else " "

        preds.append({
            "query_id": query_id,
            "predicted_citations": citations_str,
        })

        if provider == "openai": 
            time.sleep(5)  # Sleep to respect rate limits

    pred_df = pd.DataFrame(preds)
    return pred_df

In [9]:
pred_df = generate_pred_df(valid_df, chat_model, prompt, provider)
pred_df.head(5)

Generating predictions:   0%|          | 0/10 [00:00<?, ?it/s]

,query_id,predicted_citations
0,val_001,Art. 221 Abs. 1
1,val_002,
2,val_003,
3,val_004,
4,val_005,


In [10]:
from uretriever.citation import parse_citations
from uretriever.metrics import macro_f1


merged_df = pd.merge(pred_df, valid_df, on="query_id", how="inner")

preds = [
    parse_citations(row.get("predicted_citations", "")) 
    for _, row in merged_df.iterrows()
]
golds = [
    parse_citations(row.get("gold_citations", "")) 
    for _, row in merged_df.iterrows()
]

scores = macro_f1(preds, golds)
print("\n-- Scores --\n")
print(scores)


-- Scores --

{'macro_precision': 0.1, 'macro_recall': 0.005555555555555555, 'macro_f1': 0.010526315789473684}


# 5. Create Submission

In [11]:
# import pandas as pd

# test_df = pd.read_csv(DATASET_DIR / "raw" / "test.csv")
# test_df.head()

In [12]:
# test_pred_df = generate_pred_df(valid_df, chat_model, prompt, provider)
# test_pred_df.head(5)